# CAMEL CBT Standalone Kaggle Inference Server

Chỉ load `cactus-camel/camel-llama3` và expose `/v1/generate` qua ngrok. Notebook này là CBT counseling baseline độc lập, không gọi backend therapy system.


In [1]:
# 1. Install dependencies. Run once, then restart runtime if packages changed.
!pip install -q -U "transformers>=4.45.0" "accelerate>=0.26.0" bitsandbytes pyngrok fastapi uvicorn nest_asyncio sentencepiece protobuf einops


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 69.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, wh

In [2]:
# 2. Runtime setup
import os

# Dual GPU by default for Kaggle T4 x2. Override before this cell if needed.
os.environ.setdefault("CAMEL_VISIBLE_DEVICES", "0,1")
visible_devices = os.getenv("CAMEL_VISIBLE_DEVICES", "").strip()
if visible_devices:
    os.environ["CUDA_VISIBLE_DEVICES"] = visible_devices
    print(f"CUDA_VISIBLE_DEVICES={visible_devices}")

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("PYTORCH_ALLOC_CONF", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

import gc
import re
import torch
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

HF_TOKEN = os.getenv("HF_TOKEN", "").strip() or os.getenv("HUGGINGFACE_TOKEN", "").strip()
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Hugging Face token loaded.")
    except Exception as exc:
        print("WARNING: Hugging Face login failed:", repr(exc))


def gpu_mem_info(index: int):
    try:
        free, total = torch.cuda.mem_get_info(index)
    except TypeError:
        with torch.cuda.device(index):
            free, total = torch.cuda.mem_get_info()
    return free / 1024**3, total / 1024**3

if not torch.cuda.is_available():
    print("WARNING: GPU is not enabled. Turn on GPU in Kaggle Accelerator settings.")
else:
    print(f"CUDA GPUs visible: {torch.cuda.device_count()}")
    for gpu_idx in range(torch.cuda.device_count()):
        free_gb, total_gb = gpu_mem_info(gpu_idx)
        print(f"GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)} | free/total {free_gb:.2f}GB / {total_gb:.2f}GB")


CUDA_VISIBLE_DEVICES=0,1
CUDA GPUs visible: 2
GPU 0: Tesla T4 | free/total 14.46GB / 14.56GB
GPU 1: Tesla T4 | free/total 14.46GB / 14.56GB


In [3]:
# 3. Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = os.getenv("CAMEL_MODEL_NAME", "cactus-camel/camel-llama3")
GPU_MEMORY_LIMIT = os.getenv("CAMEL_GPU_MEMORY", "12GiB,12GiB")
CPU_MEMORY_LIMIT = os.getenv("CAMEL_CPU_MEMORY", "32GiB")
ALLOW_FP16_FALLBACK = os.getenv("CAMEL_ALLOW_FP16_FALLBACK", "0").strip() == "1"
print(f"Loading {MODEL_NAME}...")


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def build_max_memory_map(gpu_memory_limit: str, cpu_memory_limit: str):
    if not torch.cuda.is_available():
        return None
    gpu_count = torch.cuda.device_count()
    values = [item.strip() for item in gpu_memory_limit.split(",") if item.strip()]
    if not values:
        values = ["12GiB"]
    if len(values) == 1:
        per_gpu = {idx: values[0] for idx in range(gpu_count)}
    elif len(values) >= gpu_count:
        per_gpu = {idx: values[idx] for idx in range(gpu_count)}
    else:
        raise ValueError(
            f"CAMEL_GPU_MEMORY has {len(values)} value(s) for {gpu_count} GPU(s). "
            "Use one shared value like 12GiB or one value per GPU like 12GiB,12GiB."
        )
    per_gpu["cpu"] = cpu_memory_limit
    return per_gpu

try:
    del model
except NameError:
    pass
cleanup_cuda()
MAX_MEMORY_MAP = build_max_memory_map(GPU_MEMORY_LIMIT, CPU_MEMORY_LIMIT)
if torch.cuda.is_available():
    print(f"CUDA GPUs visible at load: {torch.cuda.device_count()}")
    for gpu_idx in range(torch.cuda.device_count()):
        free_gb, total_gb = gpu_mem_info(gpu_idx)
        print(f"GPU {gpu_idx} free/total at load start: {free_gb:.2f}GB / {total_gb:.2f}GB")
    print("Accelerate max_memory map:", MAX_MEMORY_MAP)

auth_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True, **auth_kwargs)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

OFFLOAD_DIR = "/kaggle/working/camel_offload"
os.makedirs(OFFLOAD_DIR, exist_ok=True)
base_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
    "low_cpu_mem_usage": True,
    "offload_folder": OFFLOAD_DIR,
    **auth_kwargs,
}
if MAX_MEMORY_MAP is not None:
    base_kwargs["max_memory"] = MAX_MEMORY_MAP


def quant_4bit_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )


def quant_8bit_config():
    return BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )

load_attempts = []
if torch.cuda.is_available():
    load_attempts.extend([
        ("4-bit", {**base_kwargs, "quantization_config": quant_4bit_config()}),
        ("8-bit CPU-offload", {**base_kwargs, "quantization_config": quant_8bit_config()}),
    ])
if ALLOW_FP16_FALLBACK or not torch.cuda.is_available():
    load_attempts.append(("fp16 fallback", dict(base_kwargs)))
else:
    print("Skipping fp16 fallback on GPU. Set CAMEL_ALLOW_FP16_FALLBACK=1 only on larger GPUs.")

errors = []
model = None
for label, kwargs in load_attempts:
    cleanup_cuda()
    try:
        print("Trying:", label)
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
        print("Loaded with:", label)
        break
    except Exception as exc:
        errors.append(f"{label}: {type(exc).__name__}: {exc}")
        print(label, "failed:", repr(exc))
        try:
            del model
        except Exception:
            pass
        model = None
        cleanup_cuda()

if model is None:
    raise RuntimeError(
        "Cannot load model after all safe attempts.\n"
        "Restart runtime, run cells from top, and for dual T4 use CAMEL_GPU_MEMORY=12GiB,12GiB.\n"
        + "\n".join(errors)
    )

model.eval()
print("Model ready.")
print("hf_device_map:", getattr(model, "hf_device_map", None))


Loading cactus-camel/camel-llama3...
CUDA GPUs visible at load: 2
GPU 0 free/total at load start: 14.46GB / 14.56GB
GPU 1 free/total at load start: 14.46GB / 14.56GB
Accelerate max_memory map: {0: '12GiB', 1: '12GiB', 'cpu': '32GiB'}


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

Loaded with: 4-bit
Model ready.
hf_device_map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [4]:
# 4. Standalone FastAPI server
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import traceback

app = FastAPI(title="CAMEL CBT Standalone Inference")

SYSTEM_PROMPT = """Bạn là một trợ lý tư vấn theo CBT cho sinh viên Việt Nam. BẮT BUỘC trả lời 100% bằng tiếng Việt tự nhiên. Không dùng tiếng Anh. Không dịch lại đề bài. Dùng xác nhận cảm xúc, câu hỏi gợi mở Socratic/ABC khi phù hợp, không chẩn đoán, không đưa chỉ dẫn y tế. Nếu có nguy cơ tự hại, ưu tiên an toàn và khuyến khích liên hệ người thật/cấp cứu."""
PROMPT_STYLE = "llama3"
PROMPT_LABEL = "CAMEL"


def parse_benchmark_prompt(prompt: str):
    prompt = str(prompt or "").strip()
    if "\nUser:" not in prompt:
        return prompt, []

    before_user, after_user = prompt.rsplit("\nUser:", 1)
    query = after_user.replace(f"{PROMPT_LABEL}:", "").strip()
    history_text = before_user.replace("Lịch sử:", "").strip()
    if history_text == "(trống)":
        history_text = ""

    history = []
    if history_text:
        chunks = re.split(r"\n(?=User:|Assistant:)", history_text)
        current_user = None
        for chunk in chunks:
            chunk = chunk.strip()
            if chunk.startswith("User:"):
                current_user = chunk[len("User:"):].strip()
            elif chunk.startswith("Assistant:") and current_user is not None:
                assistant = chunk[len("Assistant:"):].strip()
                history.append((current_user, assistant))
                current_user = None
    return query, history[-6:]


def vietnamese_guardrail_text(text: str) -> str:
    return (
        "Yêu cầu bắt buộc: hãy trả lời hoàn toàn bằng tiếng Việt. "
        "Không dùng tiếng Anh. Giọng điệu ấm, ngắn, tự nhiên.\n\n"
        f"Nội dung người dùng: {text}"
    )


def build_messages(query: str, history: list[tuple[str, str]]):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user_text, assistant_text in history:
        messages.append({"role": "user", "content": vietnamese_guardrail_text(user_text)})
        messages.append({"role": "assistant", "content": assistant_text})
    messages.append({"role": "user", "content": vietnamese_guardrail_text(query)})
    return messages

def manual_chat_prompt(messages: list[dict[str, str]]) -> str:
    if PROMPT_STYLE == "qwen":
        parts = []
        for message in messages:
            parts.append(f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)
    if PROMPT_STYLE == "llama3":
        parts = ["<|begin_of_text|>"]
        for message in messages:
            parts.append(f"<|start_header_id|>{message['role']}<|end_header_id|>\n\n{message['content']}<|eot_id|>")
        parts.append("<|start_header_id|>assistant<|end_header_id|>\n\nTiếng Việt: ")
        return "".join(parts)
    return "\n".join(f"{m['role']}: {m['content']}" for m in messages) + "\nassistant:"


def encode_messages(messages: list[dict[str, str]]):
    try:
        if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
            encoded = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            if torch.is_tensor(encoded):
                return {"input_ids": encoded}
            if isinstance(encoded, dict) and "input_ids" in encoded:
                return encoded
    except Exception as exc:
        print("apply_chat_template fallback:", type(exc).__name__, repr(exc))
    prompt_text = manual_chat_prompt(messages)
    return tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=2048)


def normalize_model_inputs(encoded):
    if hasattr(encoded, "data") and isinstance(encoded.data, dict):
        encoded = encoded.data
    if not isinstance(encoded, dict):
        raise TypeError(f"Tokenizer returned unsupported type: {type(encoded).__name__}")
    if "input_ids" not in encoded:
        raise KeyError(f"Tokenizer output missing input_ids. Keys: {list(encoded.keys())}")
    model_inputs = {}
    target_device = first_model_device()
    for key in ("input_ids", "attention_mask", "token_type_ids"):
        value = encoded.get(key)
        if value is None:
            continue
        if not torch.is_tensor(value):
            value = torch.tensor(value)
        model_inputs[key] = value.to(target_device)
    return model_inputs


def first_model_device():
    device_map = getattr(model, "hf_device_map", None)
    if isinstance(device_map, dict):
        cuda_devices = sorted({dev for dev in device_map.values() if isinstance(dev, int)})
        if cuda_devices:
            return torch.device(f"cuda:{cuda_devices[0]}")
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_NAME,
        "service": "camel_cbt",
        "cuda": torch.cuda.is_available(),
        "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "max_memory": MAX_MEMORY_MAP,
        "device_map": getattr(model, "hf_device_map", None),
        "prompt_style": PROMPT_STYLE,
        "has_chat_template": bool(getattr(tokenizer, "chat_template", None)),
    }


@app.post("/v1/generate")
async def generate(request: Request):
    try:
        data = await request.json()
        prompt = str(data.get("prompt", "")).strip()
        max_new_tokens = int(data.get("max_new_tokens", 512))
        temperature = float(data.get("temperature", 0.2))
        top_p = float(data.get("top_p", 0.9))
        repetition_penalty = float(data.get("repetition_penalty", 1.05))
        if not prompt:
            return JSONResponse({"text": ""})

        query, history = parse_benchmark_prompt(prompt)
        messages = build_messages(query, history)
        with torch.no_grad():
            encoded = encode_messages(messages)
            inputs = normalize_model_inputs(encoded)
            input_length = inputs["input_ids"].shape[-1]
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                pad_token_id=tokenizer.eos_token_id,
            )
            generated = outputs[0][input_length:]
            text = tokenizer.decode(generated, skip_special_tokens=True).strip()
            if text.lower().startswith("tiếng việt:"):
                text = text.split(":", 1)[1].strip()

        return JSONResponse({"text": str(text).strip(), "model": MODEL_NAME})
    except Exception as exc:
        return JSONResponse(
            {
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc()[-3000:],
            },
            status_code=500,
        )
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [5]:
# 5. Start ngrok + uvicorn
import uvicorn

# Hardcoded on purpose for local Kaggle benchmark runs. Keep this notebook private.
NGROK_AUTH_TOKEN = "3DeIdymjhY4kEFJuR0DoUxEVKMG_4tRBvTRKgN3qwkbB9mDYw"
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("WARNING: NGROK_AUTH_TOKEN is empty.")

try:
    ngrok.kill()
except Exception as exc:
    print("ngrok.kill warning:", repr(exc))

public_url = ngrok.connect(8000, bind_tls=True).public_url
print("=" * 80)
print(f"CAMEL_NGROK_URL={public_url}")
print("Set this URL in backend .env, then benchmark will call only this standalone server.")
print("Health:", f"{public_url}/health")
print("=" * 80)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio", log_level="info")
server = uvicorn.Server(config)
await server.serve()


INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


CAMEL_NGROK_URL=https://sprinkled-autopilot-disposal.ngrok-free.dev
Set this URL in backend .env, then benchmark will call only this standalone server.
Health: https://sprinkled-autopilot-disposal.ngrok-free.dev/health


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INFO:     42.115.239.203:0 - "POST /v1/generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     42.115.239.203:0 - "POST /v1/generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     42.115.239.203:0 - "POST /v1/generate HTTP/1.1" 200 OK


: 